# Docker Compose

## 1. What is Docker Compose?

**Docker Compose** is a tool used to define and run **multiple Docker containers/services together** using a single YAML file.

Instead of manually running:

```bash
docker run ...
docker run ...
docker run ...
```

we can describe the entire application in:

```text
docker-compose.yml
```

and then start everything with:

```bash
docker compose up
```

---

# 2. Why Do We Need Docker Compose?

Suppose we have a Flask application that needs:

- Flask → application
- Redis → caching/database
- MySQL → database

Without Docker Compose, we would have to create and configure each container separately.

```text
                Application
                    │
          ┌─────────┴─────────┐
          ↓                   ↓
       Flask                 Redis
          │
          ↓
        MySQL
```

With Docker Compose, we describe all of these services in one file:

```text
docker-compose.yml
        │
        ├── Flask
        ├── Redis
        └── MySQL
```

Then:

```bash
docker compose up
```

starts the whole application.

---

# 3. Docker Compose Architecture

For our example:

```text
                    Docker Compose
                         │
             ┌───────────┼───────────┐
             │           │           │
             ↓           ↓           ↓
           Flask        Redis       MySQL
          container    container    container
             │           │           │
             └───────────┼───────────┘
                         │
                    Docker Network
```

Docker Compose automatically creates a network for the services.

This means the containers can communicate with each other using their **service names**.

For example:

```text
Flask → redis
Flask → mysql
```

We don't need to manually find the container IP addresses.

---

# 4. Docker Compose File

The default file is usually:

```text
docker-compose.yml
```

or:

```text
compose.yml
```

Example:

```yaml
services:

  web:
    build: .
    ports:
      - "5000:5000"

  redis:
    image: redis

  mysql:
    image: mysql
```

---

# 5. Understanding `services`

```yaml
services:
```

`services` contains all the containers/services that make up our application.

In our example:

```yaml
services:

  web:
    ...

  redis:
    ...

  mysql:
    ...
```

We have three services:

```text
web
redis
mysql
```

Each service normally represents a container.

---

# 6. The Flask/Web Service

```yaml
web:
  build: .
  ports:
    - "5000:5000"
```

## `web`

```yaml
web:
```

This is the name of our Flask service.

Other containers can use:

```text
web
```

as its hostname.

---

## `build: .`

```yaml
build: .
```

This tells Docker Compose:

> Build the Docker image using the Dockerfile in the current directory.

For example:

```text
project/
│
├── Dockerfile
├── docker-compose.yml
├── app.py
└── requirements.txt
```

When we run:

```bash
docker compose up
```

Compose finds:

```text
Dockerfile
```

and builds the Flask image.

---

# 7. Port Mapping

```yaml
ports:
  - "5000:5000"
```

The syntax is:

```text
HOST_PORT:CONTAINER_PORT
```

Therefore:

```text
5000:5000
│    │
│    └── Port inside container
│
└───── Port on our computer
```

So:

```text
Browser
   │
   │ localhost:5000
   ↓
Host computer
   │
   │ port 5000
   ↓
Flask container
   │
   │ port 5000
   ↓
Flask application
```

We can therefore open:

```text
http://localhost:5000
```

---

# 8. Redis Service

```yaml
redis:
  image: redis
```

This tells Docker Compose to create a Redis container using the Redis Docker image.

Docker will pull the image if it doesn't already exist locally.

You can think of it as:

```text
docker-compose.yml
       │
       ↓
image: redis
       │
       ↓
Redis Docker Image
       │
       ↓
Redis Container
```

---

# 9. MySQL Service

```yaml
mysql:
  image: mysql
```

This tells Docker Compose to use the MySQL Docker image.

It creates another container:

```text
MySQL Docker Image
        │
        ↓
MySQL Container
```

---

# 10. IMPORTANT: Why Do We Need Both `redis` in requirements.txt AND a Redis Container?

This is a very important concept.

You might have:

```text
requirements.txt
```

containing:

```text
Flask
redis
```

and think:

> "Didn't we already install Redis?"

**No.**

The `redis` package in `requirements.txt` is the **Python Redis client**.

It is NOT the Redis server.

---

## Python Redis Package

When our Dockerfile does:

```dockerfile
RUN pip install -r requirements.txt
```

and requirements contains:

```text
redis
```

Python installs the Redis client library.

This allows our Python application to communicate with a Redis server.

For example:

```python
import redis

client = redis.Redis(
    host="redis",
    port=6379
)
```

Think:

```text
Python Application
       │
       ↓
Python redis package
       │
       │ communicates with
       ↓
Redis Server
```

---

# 11. Redis Container

This:

```yaml
redis:
  image: redis
```

provides the **actual Redis server**.

So we have:

```text
┌─────────────────────────────┐
│       Flask Container       │
│                             │
│  app.py                     │
│  Flask                      │
│  redis Python package       │
│                             │
│      Redis CLIENT           │
└──────────────┬──────────────┘
               │
               │ Redis protocol
               ↓
┌─────────────────────────────┐
│       Redis Container       │
│                             │
│      Redis SERVER            │
│                             │
│        Port 6379             │
└─────────────────────────────┘
```

### Remember:

```text
redis in requirements.txt
        ↓
Python Redis CLIENT

redis Docker image
        ↓
Redis SERVER
```

A simple analogy:

```text
Python redis package = Phone

Redis container       = Person you're calling
```

Having a phone doesn't mean the other person exists.

You need both.

---

# 12. Why Doesn't Flask Just Install Redis?

Because Flask and Redis are different applications.

Flask is our application:

```text
Flask
```

Redis is a separate server:

```text
Redis Server
```

They are therefore better represented as separate services:

```text
        Docker Compose
             │
       ┌─────┴─────┐
       ↓           ↓
     Flask        Redis
   Container    Container
```

This is one of the main ideas behind **microservice/container architecture**.

---

# 13. Communication Between Containers

Suppose Flask wants to connect to Redis.

Our Python code might be:

```python
import redis

client = redis.Redis(
    host="redis",
    port=6379
)
```

Notice:

```python
host="redis"
```

We don't use:

```python
host="localhost"
```

Why?

Because Redis is running in another container.

Docker Compose provides service-name-based networking.

Since our service is named:

```yaml
redis:
  image: redis
```

Flask can reach it using:

```text
redis
```

Therefore:

```text
Flask Container
      │
      │ host="redis"
      ↓
Redis Container
```

Similarly, MySQL can be reached using:

```text
mysql
```

---

# 14. Complete Example

Our project could look like:

```text
flask-redis-mysql/
│
├── app.py
├── requirements.txt
├── Dockerfile
└── docker-compose.yml
```

---

## `requirements.txt`

```text
Flask
redis
```

Remember:

```text
Flask
  ↓
Python web framework

redis
  ↓
Python Redis client
```

It does NOT install the Redis server.

---

# 15. Dockerfile

```dockerfile
FROM python:3.7-alpine

# Working directory inside the container
WORKDIR /code

# Flask configuration
ENV FLASK_APP=app.py
ENV FLASK_RUN_HOST=0.0.0.0

# Copy project files into /code
COPY . .

# Install Python dependencies
RUN pip install --no-cache-dir -r requirements.txt

# Flask uses port 5000
EXPOSE 5000

# Start Flask
CMD ["flask", "run"]
```

---

# 16. `docker-compose.yml`

```yaml
services:

  # Flask application
  web:
    # Build Flask image using Dockerfile
    build: .

    # HOST:CONTAINER
    ports:
      - "5000:5000"

  # Redis server
  redis:
    # Use Redis Docker image
    image: redis

  # MySQL server
  mysql:
    # Use MySQL Docker image
    image: mysql
```

---

# 17. What Happens When We Run Compose?

Run:

```bash
docker compose up
```

Docker Compose reads:

```text
docker-compose.yml
```

Then:

```text
                    docker compose up
                           │
                           ↓
                 Read docker-compose.yml
                           │
              ┌────────────┼────────────┐
              ↓            ↓            ↓
          Build web     Pull Redis    Pull MySQL
              │            │            │
              ↓            ↓            ↓
         Flask image   Redis image   MySQL image
              │            │            │
              ↓            ↓            ↓
         Flask container Redis container MySQL container
```

---

# 18. Start in Background

Normally:

```bash
docker compose up
```

runs the services in the foreground.

To run them in the background:

```bash
docker compose up -d
```

The `-d` means:

```text
detached mode
```

---

# 19. Check Running Services

```bash
docker compose ps
```

This shows the containers created by Compose.

You can also use:

```bash
docker ps
```

---

# 20. Stop the Application

```bash
docker compose down
```

This stops and removes the Compose containers and network.

It does not mean:

```text
Delete your Docker images
```

The images normally remain available.

---

# 21. Useful Docker Compose Commands

## Start services

```bash
docker compose up
```

---

## Start in background

```bash
docker compose up -d
```

---

## Stop and remove services

```bash
docker compose down
```

---

## Rebuild images

```bash
docker compose build
```

---

## Build and start

```bash
docker compose up --build
```

Useful when you changed the Dockerfile or application dependencies.

---

## See running Compose containers

```bash
docker compose ps
```

---

## View logs

```bash
docker compose logs
```

---

## View logs for one service

```bash
docker compose logs web
```

For Redis:

```bash
docker compose logs redis
```

---

## Follow logs live

```bash
docker compose logs -f
```

`-f` means follow.

It continuously displays new log output.

---

## Restart services

```bash
docker compose restart
```

---

# 22. The Difference Between Docker and Docker Compose

## Docker

Docker can run individual containers:

```bash
docker run redis
```

```bash
docker run my-flask-app
```

You manually manage the containers.

---

## Docker Compose

Compose manages a group of related containers:

```text
        docker-compose.yml
               │
       ┌───────┼────────┐
       ↓       ↓        ↓
     Flask    Redis    MySQL
```

And one command can start everything:

```bash
docker compose up
```

---

# 23. Docker Compose vs Dockerfile

These two are NOT the same thing.

## Dockerfile

A Dockerfile describes:

> **How to build one Docker image.**

Example:

```dockerfile
FROM python:3.7-alpine

WORKDIR /code

COPY . .

RUN pip install -r requirements.txt

CMD ["flask", "run"]
```

Think:

```text
Dockerfile
    ↓
Build ONE image
```

---

## Docker Compose

Docker Compose describes:

> **How multiple services/containers work together.**

Example:

```yaml
services:

  web:
    build: .

  redis:
    image: redis

  mysql:
    image: mysql
```

Think:

```text
docker-compose.yml
        ↓
Multiple services
        ↓
Flask + Redis + MySQL
```

---

# 24. Very Important Mental Model

Remember this:

```text
Dockerfile
    │
    │ builds
    ↓
Docker Image
    │
    │ creates
    ↓
Container
```

For multiple services:

```text
                  Docker Compose
                        │
          ┌─────────────┼─────────────┐
          ↓             ↓             ↓
     Dockerfile      Redis Image    MySQL Image
          ↓             ↓             ↓
     Flask Image       Redis          MySQL
          ↓             ↓             ↓
     Flask Container   Container     Container
```

So:

```text
Dockerfile
→ How to BUILD my application image

Docker Image
→ Blueprint/package for a container

Container
→ Running instance of an image

Docker Compose
→ How multiple containers/services work together
```

---

# 25. Basic Workflow to Remember

When working with a Flask + Redis application:

```text
1. Write Flask application
          ↓
2. Create requirements.txt
          ↓
3. Create Dockerfile
          ↓
4. Build Flask image
          ↓
5. Create docker-compose.yml
          ↓
6. Define Flask service
          ↓
7. Define Redis service
          ↓
8. Run docker compose up
          ↓
9. Compose creates the network
          ↓
10. Compose starts the containers
```

The final architecture:

```text
                         Your Computer
                              │
                              │
                       localhost:5000
                              │
                              ↓
                    ┌─────────────────┐
                    │ Flask Container │
                    │                 │
                    │ app.py          │
                    │ Flask           │
                    │ redis CLIENT    │
                    └────────┬────────┘
                             │
                     Docker Network
                       ┌─────┴─────┐
                       ↓           ↓
                ┌───────────┐ ┌───────────┐
                │   Redis   │ │   MySQL   │
                │ Container │ │ Container │
                │           │ │           │
                │ Redis     │ │ MySQL     │
                │ SERVER    │ │ SERVER    │
                └───────────┘ └───────────┘
```

# 26. The Most Important Thing to Remember

```text
requirements.txt
        │
        ↓
Installs Python libraries
        │
        ├── Flask
        └── redis ← Python CLIENT
                       │
                       │ communicates with
                       ↓
                 Redis Container
                       │
                       ↓
                  Redis SERVER
```

Therefore:

> **Installing `redis` with pip does NOT install the Redis server.**

You install the Python client with:

```bash
pip install redis
```

and run the actual Redis server using:

```yaml
redis:
  image: redis
```

That's why both exist in a Docker Compose application.